# 08 실제 LLM 평가와 실행 추적

작은 로컬 LLM이 실제 답변을 평가합니다. 사람이 정의한 판정과 비교하고, 검색·생성·평가를 하나의 요청 기록으로 연결합니다. 07번을 미리 실행하지 않아도 됩니다.

**준비:** Ollama 앱 실행 → 별도 터미널에서 `ollama pull qwen2.5:1.5b-instruct` → 이 파일의 Python 3.11 커널 선택.
기존 `requirements/week11-offline.txt`만으로 실행됩니다. Ollama Python SDK나 클라우드 계정은 필요 없습니다.
모델 다운로드는 최초 한 번만 필요합니다. 앱이 없거나 모델이 없으면 준비 방법을 표시하고 중단하며, 가짜 응답으로 대체하지 않습니다.

첫 호출은 모델 로딩 때문에 느릴 수 있습니다. 출력 최대 192토큰, 문맥 2,048토큰, temperature=0, 모델 유지 10분입니다.
Qwen2.5 Instruct는 추론 전용 모델이 아니며, thinking 지원 모델로 바꾸면 `think=False`를 보냅니다.
**속도와 문장은 기기·모델 버전에 따라 달라집니다. 특정 문장 또는 의미 판정의 일치를 성공 조건으로 삼지 않습니다.**

[전체 실습 순서](../WALKTHROUGH.md) / [실제 호출 구현](../ollama_lab.py)


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/ollama_lab.py").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info[:2] == (3, 11), "저장소 .venv의 Python 3.11 커널을 선택하세요."
import pandas as pd
from IPython.display import display
from week11.lab.ollama_lab import (
    OllamaClient,
    generate_one,
    judge_answer,
    save_run,
    now,
    experiment_manifest,
)
from week11.lab.evaluation import load_data

client = OllamaClient()  # 기본 qwen2.5:1.5b-instruct / 외부 API 키 불필요
print(client.probe())
docs, questions = load_data()
by_id = {q["id"]: q for q in questions}

## 1. 평가자의 실제 판정

05번에서 사용한 작성 예시를 이번에는 실제 모델에 전달합니다.

In [ ]:
from week11.lab.judging import FIXTURES

cases = [f for f in FIXTURES if f["id"] in ["J01", "J02", "J03", "J05"]]
judgments = []
for case in cases:
    judged = judge_answer(
        client, "근거가 답변의 주장을 뒷받침하는가?", case["evidence"], case["answer"]
    )
    judgments.append(
        {
            "id": case["id"],
            "expected": case["gold"],
            "actual": judged["parsed"]["verdict"],
            "reason": judged["parsed"]["reason"],
            "latency_ms": judged["latency_ms"],
        }
    )
display(pd.DataFrame(judgments))

**확인 결과**

표의 actual과 reason은 실제 모델 출력입니다. 05번에 저장된 judge 열을 재사용하지 않습니다. 불일치가 생기면 근거와 판정 기준을 다시 읽습니다.

## 2. 일치율과 불일치 원인

판정 형식이 맞아도 평가자가 틀릴 수 있습니다.

In [ ]:
frame = pd.DataFrame(judgments)
print("선택 예시 일치:", int((frame.expected == frame.actual).sum()), "/", len(frame))
display(frame[frame.expected != frame.actual])

**확인 결과**

4건의 시연 계산이며 일반화된 모델 성능이 아닙니다. 작은 모델과 자기 평가의 한계를 사례로 설명합니다.

## 3. 생성 응답의 실제 평가

Q03a를 새로 생성한 뒤 동일한 근거로 의미 평가를 요청합니다.

In [ ]:
row = generate_one(client, by_id["Q03a"])
row["judge"] = judge_answer(
    client, row["question"], [d["text"] for d in row["ranked"]], row["answer"]["answer"]
)
row["spans"].append(
    {
        "name": "llm_judge",
        "parent_id": row["trace_id"],
        "duration_ms": row["judge"]["latency_ms"],
        "result": row["judge"]["parsed"],
    }
)
print(row["answer"])
print(row["judge"]["parsed"])
print(row["checks"])

**확인 결과**

생성·코드 검사·LLM 판정을 따로 읽습니다. 같은 모델의 자기 평가는 독립 검증이 아니며 사람 검토를 대체하지 않습니다.

## 4. 요청과 단계별 기록

trace는 전체 요청, span은 요청 안의 개별 작업입니다.

In [ ]:
print("trace_id:", row["trace_id"])
display(pd.DataFrame(row["spans"]))
print("모델:", client.info["model"], "digest:", client.info["digest"])
print("프롬프트 해시:", row["generation"]["prompt_sha256"])

**확인 결과**

동일 trace_id 아래 retrieval·generation·code_checks·llm_judge가 연결됩니다. 이 파일의 JSON은 교육용 로컬 추적 형식이며 OpenTelemetry 전송 형식 자체는 아닙니다.

## 5. Langfuse·LangSmith와의 연결

플랫폼은 이 기록을 팀이 검색·비교·검토할 수 있게 관리합니다.

In [ ]:
display(
    pd.DataFrame(
        [
            {
                "실습 기록": "trace_id + spans",
                "Langfuse": "Trace / observations",
                "LangSmith": "Run trace / child runs",
            },
            {
                "실습 기록": "question + reference",
                "Langfuse": "Dataset item",
                "LangSmith": "Dataset example",
            },
            {
                "실습 기록": "prompt_version + model",
                "Langfuse": "Experiment",
                "LangSmith": "Experiment",
            },
            {
                "실습 기록": "checks + judge + 사람 검토",
                "Langfuse": "Scores / annotation",
                "LangSmith": "Feedback / annotation queue",
            },
        ]
    )
)

**확인 결과**

클라우드나 자체 호스팅 전송은 수행하지 않습니다. 플랫폼 도입은 선택이지만 입력·출력·버전·지연·평가 근거의 기록은 필요합니다. 실제 계정 연동은 공식 SDK 안내를 따릅니다.

## 6. 검토 가능한 결과 저장

질문 하나의 근거와 평가 판정을 함께 보관합니다.

In [ ]:
result = {
    "created_at": now(),
    "provenance": "real local Ollama generation and judge; not a benchmark",
    **experiment_manifest(client),
    "rows": [row],
    "judge_calibration": judgments,
}
report = save_run(result, ROOT / "week11/lab/reports/judge")
print(report)
print("macOS: open week11/lab/reports/judge/ollama-report.html")

**확인 결과**

judge 폴더에 평가자 판정과 요청 추적을 담은 HTML·JSON·JSONL이 생성됩니다. API 키 없이 로컬에서 끝납니다.

## 다음 실험

질문·프롬프트·모델 중 하나만 바꾸고 응답, 근거, 지연을 비교합니다. 이 사례는 개발용 시연 질문이며 미공개 test 벤치마크가 아닙니다.
생성 결과와 자체 평가가 모두 좋아도 사람의 의미 검토가 필요합니다. 실제 사용자 정보나 API 키를 입력·공개하지 않습니다.

연결 실패: Ollama 앱/`ollama serve`. 모델 없음: `ollama list` 후 pull. 긴 대기: 최초 로드 시간과 생성 시간을 분리해 확인.
변수가 섞이면 **Restart Kernel → Run All**. 출력은 Git 제외 `week11/lab/reports/`에만 저장합니다.
